# 06 — Methodenvergleich (Schritt 11)

Drei Methoden im Vergleich: **`zscore_stl`** (Punkt-Outlier auf STL-Residual),
**`arima`** (lokal-prognostische Abweichung pro Peer-Cluster) und
**`cluster_segment`** (Form-/Segment-untypisch, native Segment-Tag-Granularität).
Der Autoencoder ist aus dem Vergleich genommen (Stage-A-Befund:
macOS-Hang in `model.fit` auf realer Datengröße; Modul + Driver bleiben für
Linux/CI im Code — siehe `reports/methodology.md`).

Vorgehen:

1. **Cluster-Anker** — Test-Flag-Rate aus dem aktuellen
   `anomaly_scores.parquet` ableiten (nicht hartcodiert).
2. **X-Sweep** der Anteils-Aggregation auf `(site, date, segment)` —
   `aggregate_to_segment_day(threshold_pct=…)` über
   `{0.0, 0.10, 0.25, 0.50, 0.75}`. `0.0` reproduziert das alte „any".
3. **X_default wählen** — kleinster X, bei dem die Punkt-Methoden in
   die Größenordnung Cluster-Anker fallen (Ratio 0,5–2).
4. **κ-Heatmap** über mehrere X — bleibt Komplementarität erhalten?
5. **Inferenzzeit** auf 5 Sites (Loader-Lauf einmal).
6. **Precision** aus `reports/annotation/annotation.csv` — heute noch leer;
   Notebook läuft trotzdem durch, Empfehlung ist solange „Ensemble bis
   Annotation". Sobald Felix & Jakob ihre Labels einspielen, re-evaluiert
   `recommend_strategy()` ohne Code-Änderung.
7. **Vergleichstabelle** als Markdown in `reports/tables/06_method_comparison.md`.

Nach diesem Lauf: `X_default` in `config/config.yaml`
(`comparison.aggregation_threshold_pct`) und in `reports/methodology.md`
festschreiben.

In [1]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from rausch_energy_anomaly.evaluation.method_comparison import (
    DEFAULT_TRAIN_END,
    EXCLUDE_SITES,
    aggregate_to_segment_day,
    compare_at_thresholds,
    inference_timing,
    load_default_threshold_pct,
    precision_from_annotation,
    recommend_strategy,
    summary_table,
)

_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
print("project root:", _ROOT)

scores = pd.read_parquet(_ROOT / "data" / "processed" / "anomaly_scores.parquet")
print(
    f"scores: {len(scores)} rows, methods: {sorted(scores['method'].unique())}, "
    f"sites: {scores['site'].nunique()}"
)

project root: /Users/jakobringel/Projekt/datenwerkios-anomalie
scores: 6745108 rows, methods: ['arima', 'autoencoder', 'cluster_segment', 'zscore_stl'], sites: 23


## 1 — Cluster-Anker: Test-Flag-Rate aus dem aktuellen Parquet

`cluster_segment` ist nativ Segment-Tag-granular; ihre Test-Flag-Rate
ist der Anker für die X-Wahl der Punkt-Methoden — direkt aus dem Parquet
abgeleitet, nicht aus dem Smoke-Lauf erinnert.

In [2]:
ts_all = pd.to_datetime(scores["timestamp"])
scores_ex23 = scores[~scores["site"].isin(EXCLUDE_SITES)].copy()
cluster_rows = scores_ex23[scores_ex23["method"] == "cluster_segment"].copy()
cluster_rows["date"] = pd.to_datetime(cluster_rows["timestamp"]).dt.date
cluster_test = cluster_rows[cluster_rows["date"] > DEFAULT_TRAIN_END]
cluster_anchor = float(cluster_test["flag"].astype(bool).mean())
print(f"Cluster-Distanz Test-Flag-Rate: {cluster_anchor * 100:.2f}%  (n={len(cluster_test)})")

Cluster-Distanz Test-Flag-Rate: 0.64%  (n=32696)


## 2 — X-Sweep: Flag-Raten je Methode (Train / Test)

`compare_at_thresholds` aggregiert die Punkt-Methoden je X auf Segment-Tag
und berechnet Flag-Rate (Train ≤ 2024, Test 2025+). Cluster-Distanz ist X-
invariant (nativ Segment-Tag).

In [3]:
SWEEP_X = (0.0, 0.10, 0.25, 0.50, 0.75)
flag_rates, pairwise = compare_at_thresholds(scores, thresholds=SWEEP_X)
flag_rates_pivot = flag_rates.pivot(
    index="threshold_pct", columns="method", values="flag_rate_test"
)
print("Flag-Rate Test je Methode × X:")
print((flag_rates_pivot * 100).round(3).to_string())

Flag-Rate Test je Methode × X:
method          arima  autoencoder  cluster_segment  zscore_stl
threshold_pct                                                  
0.00           28.639       14.247            0.641      13.129
0.10            8.713        5.015            0.641       7.135
0.25            1.048        1.208            0.641       3.898
0.50            0.052        0.120            0.641       1.802
0.75            0.000        0.009            0.641       0.634


In [4]:
fig, ax = plt.subplots(figsize=(9, 5))
colors = {"zscore_stl": "tab:blue", "arima": "tab:orange", "cluster_segment": "tab:green"}
for method in flag_rates_pivot.columns:
    ax.plot(
        flag_rates_pivot.index,
        flag_rates_pivot[method] * 100,
        marker="o",
        label=method,
        color=colors.get(method),
    )
ax.axhline(
    cluster_anchor * 100,
    color="gray",
    linestyle="--",
    linewidth=0.8,
    label=f"cluster anchor ({cluster_anchor * 100:.2f}%)",
)
ax.set_xlabel("threshold_pct (Anteil flagged points im Segment)")
ax.set_ylabel("Flag-Rate Test [%]")
ax.set_yscale("log")
ax.set_title("X-Sweep — Test-Flag-Rate je Methode")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
out = _ROOT / "reports" / "figures" / "06_sweep_flag_rates.png"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out, dpi=120)
plt.show()
print(f"saved: {out.relative_to(_ROOT)}")

saved: reports/figures/06_sweep_flag_rates.png


/var/folders/6x/0jtnv68j031_8bd1s9hjbnrw0000gn/T/ipykernel_60198/3921008102.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 — X-Default-Wahl

Kriterium: **kleinster X, bei dem `zscore_stl` UND `arima` Test-Flag-Rate
zwischen 0,5× und 2× des Cluster-Ankers liegen** — vergleichbare Detektor-
Sensitivität, kein „Aggregations-Artefakt mehr". Zu klein → die any-
Aufblähung dominiert (arima ~30 %, zscore ~14 % am alten Smoke-Lauf).
Zu groß → die Methoden flaggen kaum mehr etwas, Vergleich verarmt.

In [5]:
def _ratio(rate):
    return rate / cluster_anchor if cluster_anchor > 0 else float("nan")


cand_rows = []
for x in [x for x in SWEEP_X if x > 0]:
    row = flag_rates_pivot.loc[x]
    zs = float(row.get("zscore_stl", float("nan")))
    ar = float(row.get("arima", float("nan")))
    ratios = [_ratio(zs), _ratio(ar)]
    in_band = all(0.5 <= r <= 2.0 for r in ratios)
    cand_rows.append(
        {
            "threshold_pct": x,
            "zscore_test_%": zs * 100,
            "arima_test_%": ar * 100,
            "zs_ratio_to_anchor": ratios[0],
            "ar_ratio_to_anchor": ratios[1],
            "in_band": in_band,
        }
    )
cand_df = pd.DataFrame(cand_rows)
print(cand_df.round(3).to_string(index=False))

in_band = cand_df[cand_df["in_band"]]
if len(in_band):
    heuristic_x = float(in_band["threshold_pct"].min())
    heuristic_rationale = (
        f"Heuristik: X={heuristic_x} ist der kleinste Sweep-Wert, bei dem zscore_stl "
        f"UND arima in 0,5×..2× des Cluster-Ankers ({cluster_anchor * 100:.2f}%) liegen."
    )
else:
    # Heuristik: kleinster Abstand zur Anker-Rate (fallback)
    cand_df["max_distance_to_band"] = cand_df.apply(
        lambda r: max(
            abs(r["zs_ratio_to_anchor"] - 1.0),
            abs(r["ar_ratio_to_anchor"] - 1.0),
        ),
        axis=1,
    )
    heuristic_x = float(cand_df.loc[cand_df["max_distance_to_band"].idxmin(), "threshold_pct"])
    heuristic_rationale = (
        f"Heuristik: kein Sweep-Wert in [0,5×..2×]-Band; nächst-bester nach Anker-Nähe "
        f"wäre X={heuristic_x} — ABER die Heuristik berücksichtigt nicht, dass dabei "
        "eine Methode (ARIMA) auf 0 % Flag-Rate kollabiert."
    )
print(heuristic_rationale)

# Override: methodische Entscheidung aus config.yaml (post-Sweep-Wahl, vgl. methodology.md).
config_x = load_default_threshold_pct()
if config_x is not None:
    chosen_x = float(config_x)
    rationale = (
        f"X_default = {chosen_x} aus config.yaml (comparison.aggregation_threshold_pct). "
        "Methodische Entscheidung nach dem Sweep: ARIMA-Sichtbarkeit erhalten "
        f"(Ratio 1,64 zum Cluster-Anker bei X=0,25). Die Z-Score/ARIMA-Asymmetrie "
        "(zscore_test 3,90 % vs. ARIMA 1,05 % vs. Cluster 0,64 % — kein gemeinsames X "
        "kalibriert beide Punkt-Methoden auf dieselbe Flag-Rate) wird als "
        "eigenständiger Befund ausgewiesen."
    )
else:
    chosen_x = heuristic_x
    rationale = (
        heuristic_rationale + " (Kein Override in config.yaml gesetzt — Wahl noch offen.)"
    )

print(f"\n→ X_default = {chosen_x}")
print(rationale)

 threshold_pct  zscore_test_%  arima_test_%  zs_ratio_to_anchor  ar_ratio_to_anchor  in_band
          0.10          7.135         8.713              11.161              13.630    False
          0.25          3.898         1.048               6.098               1.640    False
          0.50          1.802         0.052               2.819               0.082    False
          0.75          0.634         0.000               0.992               0.000    False
Heuristik: kein Sweep-Wert in [0,5×..2×]-Band; nächst-bester nach Anker-Nähe wäre X=0.75 — ABER die Heuristik berücksichtigt nicht, dass dabei eine Methode (ARIMA) auf 0 % Flag-Rate kollabiert.

→ X_default = 0.25
X_default = 0.25 aus config.yaml (comparison.aggregation_threshold_pct). Methodische Entscheidung nach dem Sweep: ARIMA-Sichtbarkeit erhalten (Ratio 1,64 zum Cluster-Anker bei X=0,25). Die Z-Score/ARIMA-Asymmetrie (zscore_test 3,90 % vs. ARIMA 1,05 % vs. Cluster 0,64 % — kein gemeinsames X kalibriert beide Punkt-Methode

## 4 — κ-Heatmap über Sweep

Sinkt κ bei höherem X gegen 0 → Methoden werden disjunkt → Ensemble-Pfad
sinnvoll. Bleibt κ hoch → Methoden konvergieren → Sieger-Pfad denkbar,
sofern auch Precision differenziert.

In [6]:
def kappa_heatmap(pairwise, x, ax):
    sub = pairwise[pairwise["threshold_pct"] == x]
    methods = sorted(set(sub["method_a"]) | set(sub["method_b"]))
    mat = pd.DataFrame(np.nan, index=methods, columns=methods)
    for _, row in sub.iterrows():
        mat.loc[row["method_a"], row["method_b"]] = row["kappa"]
        mat.loc[row["method_b"], row["method_a"]] = row["kappa"]
    for m in methods:
        mat.loc[m, m] = 1.0
    sns.heatmap(
        mat.astype(float),
        annot=True,
        fmt=".2f",
        vmin=-0.2,
        vmax=1.0,
        cmap="RdBu_r",
        center=0.4,
        ax=ax,
        cbar=False,
    )
    ax.set_title(f"κ bei X = {x}")


fig, axes = plt.subplots(1, len(SWEEP_X), figsize=(4.2 * len(SWEEP_X), 4))
for ax, x in zip(axes, SWEEP_X, strict=False):
    kappa_heatmap(pairwise, x, ax)
fig.tight_layout()
out = _ROOT / "reports" / "figures" / "06_kappa_heatmap.png"
fig.savefig(out, dpi=120)
plt.show()
print(f"saved: {out.relative_to(_ROOT)}")

saved: reports/figures/06_kappa_heatmap.png


/var/folders/6x/0jtnv68j031_8bd1s9hjbnrw0000gn/T/ipykernel_60198/1618704423.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 — Inferenzzeit (5 Sites, ein Loader-Lauf)

Loader-Pass einmal (~45 s), pro Methode wird `fit + score` gemessen.
**Wird einmalig ausgeführt** und in die Vergleichstabelle übernommen —
kein erneuter Lauf bei Notebook-Re-Renders. ARIMA ist der dominante
Block (auto_arima 3× über die Peer-Gruppen).

In [7]:
timing = inference_timing(category="Baumärkte")
print(timing.to_string(index=False))

/Users/jakobringel/Projekt/datenwerkios-anomalie/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


         method  wall_time_fit_s  wall_time_score_s  n_sites
     zscore_stl         0.001340           0.000358        5
          arima       118.357286          50.061387        5
cluster_segment         1.429602           0.008554        5
    autoencoder         5.212632           3.203254        5


## 6 — Precision aus Annotation

Strenge Lesart (siehe `precision_from_annotation`-Docstring):
`plausibel_anomal → TP`, `erklärbar → FP`, `unklar → ausgeschlossen`. Solange
die `label`-Spalten leer sind, liefert die Funktion einen leeren DataFrame
und die Empfehlung bleibt „Ensemble bis Annotation".

In [8]:
precision = precision_from_annotation()
if len(precision):
    print(precision.to_string(index=False))
else:
    print("(noch keine Annotationen — Felix & Jakob füllen heute Abend)")

         method  n_labeled  tp  fp  unklar  precision
          arima         17  17   0       0        1.0
cluster_segment         20  20   0       0        1.0
     zscore_stl         20  20   0       0        1.0


## 7 — Vergleichstabelle + Empfehlung

In [9]:
md_table = summary_table(
    flag_rates, pairwise, x_default=chosen_x, timing_df=timing, precision_df=precision
)
print(md_table)

tables_dir = _ROOT / "reports" / "tables"
tables_dir.mkdir(parents=True, exist_ok=True)
out = tables_dir / "06_method_comparison.md"
out.write_text(
    f"# Methodenvergleich (Schritt 11) — X_default = {chosen_x}\n\n"
    f"Cluster-Anker (Test-Flag-Rate): {cluster_anchor * 100:.2f}%\n\n"
    f"X-Wahl-Rationale: {rationale}\n\n"
    f"{md_table}\n",
    encoding="utf-8",
)
print(f"\nsaved: {out.relative_to(_ROOT)}")

| Methode | Native Granularität | Flag-Rate Train | Flag-Rate Test | κ vs andere | Wall-Time fit (s) | Wall-Time score (s) | Precision | Stärke | Schwäche |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| arima | point (15 min) | 1.03% | 1.05% | autoencoder=0.11, cluster_segment=0.02, zscore_stl=0.08 | 118.36 | 50.06 | 100% | Lokal-prognostische Abweichung, Peer-Gruppen-Sensitivität | Sensitiv gegen Train-Bias, langsamer |
| autoencoder | point (15 min) | 0.89% | 1.21% | arima=0.11, cluster_segment=0.03, zscore_stl=0.03 | 5.21 | 3.20 | — | Form-/Niveau-Abweichung im 24h-Lastgang; pro-Site-normiert, deep | DST-/Teiltage werden NaN; Zwischen-Site-Magnitude wegnormiert |
| cluster_segment | segment_day | 1.01% | 0.64% | arima=0.02, autoencoder=0.03, zscore_stl=-0.01 | 1.43 | 0.01 | 100% | Form-/Segment-untypisch; methoden-agnostische Diagnose | Keine 15-min-Lokalisierung im Segment |
| zscore_stl | point (15 min) | 5.06% | 3.90% | arima=0.08, autoencoder=0.03, cluster_segm

In [10]:
strategy, label, rationale_rec = recommend_strategy(
    precision, pairwise, x_default=chosen_x
)
print(f"strategy   = {strategy}")
print(f"label      = {label}")
print(f"rationale  = {rationale_rec}")

strategy   = ensemble
label      = union
rationale  = 3 Methoden erfüllen das Sieger-Kriterium (arima, cluster_segment, zscore_stl); ihre paarweise κ ≤ 0,40 zeigt, dass sie disjunkte Anomalie-Mengen detektieren. Union summiert komplementäres Wissen — Default-Ensemble für das Dashboard.


## 8 — Was als Nächstes ins Repo geht

- `config/config.yaml` — `comparison.aggregation_threshold_pct: <chosen_x>` ablegen.
- `reports/methodology.md` — Befund-Abschnitt „Methodenvergleich (Schritt 11)"
  ergänzen: Sweep-Tabelle, X-Wahl-Begründung, Strategie-Empfehlung, Verweis
  auf `reports/tables/06_method_comparison.md` und die zwei Figuren.
- Nach Annotation: dieses Notebook neu ausführen — Precision wird automatisch
  eingelesen, `recommend_strategy` re-evaluiert.